# Dembrow, Sawchuk et al. (2024) on dandiset 001065

Paper. Dembrow NC, Sawchuk S, et al. (2024). Areal specializations in the morpho-electric and
transcriptomic properties of primate layer 5 extratelencephalic projection neurons. Cell Reports
43:114718. doi:10.1016/j.celrep.2024.114718.

Dataset. DANDI:001065, version 0.241028.1559 (doi:10.48324/dandi.001065/0.241028.1559): 294
MIES current clamp NWB files, one per neuron, from macaque temporal and motor cortex. The paper
cites the dandiset without a version; 0.241028.1559 is its only published version (the draft has
the same 294 assets), so it is pinned.

Claims. `claims/001065/10.1016-j.celrep.2024.114718.yaml`, written from the accepted author
manuscript (PMC11488157) with its STAR Methods and supplement, and from the paper's code (Zenodo
record 13749343). The author added one link claim, `link-cells`.

Outside inputs. The files hold no area, preparation or subclass labels. They come from the
per-cell tables of the paper's code on Zenodo (doi:10.5281/zenodo.13749343): area, preparation,
transcriptomic, retrograde and predicted subclass, and drug, from `L5_physiology_121223.csv`; the
cells of the 4 nA protocol from the keys of `MCx.pkl` and `TCx.pkl`. Their feature values are used
only to check the link. The morphology claims use the dendritic features of `RawFeatureWide.csv`
(same record), as the reconstructions themselves are in the Brain Image Library.

Every feature is computed here from the voltage and command traces (helper `dembrow.py`); the
paper's feature extraction code was not deposited.

| Claim | Tested here |
|---|---|
| link-cells (author's link claim) | tested |
| cohort-animals, cohort-recordings | tested |
| patchseq-qc-177, cs-map-to-l5et, fig4h-patchseq-genes-et, patchseq-genes-area | blocked: Patch-seq reads in SRA, not processed here |
| fig1-snrna-genes-et, fig1-snrna-genes-area | blocked: snRNA-seq references in NeMO, not processed here |
| fig3a-cs-with-l5et, fig3e-classifier-accuracy, fig3-spc-classifier | tested |
| fig3h-projection-largest-effect, fig4a, fig4c, fig4e, fig4g, fig4-interaction-small | tested |
| species-no-effect, tcx-et-types-no-difference | tested |
| fig5-tap-root-dendrites, fig5c-basal-apical-features, fig5e-morph-classifier | tested on the paper's dendritic features |
| fig5b-dendrite-depth | blocked: needs layer-aligned reconstructions |
| fig6a, fig6b, fig6d, figs6b (three claims), fig6h | tested |
| figs6c, figs6d, fig6-depolarization-block, fig6j, fig6k | tested |
| figs7-cv-isi-higher-mcx, fig7-oscillation-frequency-vm, fig7k, figs7-oscillations-with-blockers, fig7l, figs7g | tested |

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.api  # noqa: F401, imported before the filter below, as it sets its own
import dembrow as d

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
SEED = 20241028  # each claim section makes its own generator from this seed
warnings.filterwarnings("ignore")  # statsmodels rank warnings for sparse factor cells, sklearn convergence

## Loading

Every file is streamed from the archive and reduced, as it loads, to the features of each cell
(the helper `dembrow.py` says where each definition comes from), the 1 s FI curve, the series of
1 s steps to 4 nA where there is one, and the spike times and interspike segments of the 10 s
steps. Labels come from the paper's per-cell tables on Zenodo (record 13749343), matched by the
file's `cell_id`.

In [ ]:
t0 = time.time()
assets = d.list_assets()
print(len(assets), "assets,", round(assets["size"].sum() / 1e9, 2), "GB, version", d.VERSION)
res = d.load_all(assets)
print("loaded in", round((time.time() - t0) / 60, 1), "min; files that failed:",
      [(x["path"], x["err"]) for x in res if x["err"]])

In [ ]:
SPECIES = {"http://purl.obolibrary.org/obo/NCBITaxon_9545": "M.nemestrina",
           "http://purl.obolibrary.org/obo/NCBITaxon_9544": "M.mulatta"}
ok = [x for x in res if not x["err"]]
meta = pd.DataFrame([dict(path=x["path"], **{k: v for k, v in x["meta"].items() if k != "rates"},
                          rates=tuple(x["meta"]["rates"])) for x in ok])
meta["species_name"] = meta["species"].map(SPECIES)
feats = pd.DataFrame([dict(cell_name=x["meta"]["cell_id"], **x["feats"]) for x in ok])
# the code fills missing phase features: Iphase 0, phase0 0.5 (UMAPprojections_020724.ipynb)
feats["Iphase"] = feats["Iphase"].fillna(0)
feats["phase0"] = feats["phase0"].fillna(0.5)
tab = d.zenodo_csv("L5_physiology_121223.csv")
lock = d.zenodo_csv("lockdown_physiology_072623.csv")
LAB = ["cell_name", "Donor", "Species", "slice_type", "Brain_Region", "Cortical_area",
       "predicted_subclass", "projection_class", "subclass_Corr_label", "projection_type",
       "cluster_Corr_label_MTG", "drug", "Virus", "Reporter_positive"]
cells = (feats.merge(meta[["cell_id", "subject_id", "species_name", "sex", "path"]],
                     left_on="cell_name", right_on="cell_id")
              .merge(tab[LAB], on="cell_name", how="left"))
cells["Prep"] = cells["slice_type"].str.replace("NHP_", "")
cells["area_from_region"] = np.where(cells["Brain_Region"].isin(d.MCX_REGIONS), "MCx", "TCx")
FL = d.FLIST
print(len(cells), "cells with features;", cells["Cortical_area"].notna().sum(), "with labels")

## Data check

In [ ]:
print("subjects:", meta["subject_id"].nunique(), "| files:", len(meta),
      "| files per subject: median", meta.groupby("subject_id").size().median(),
      "range", meta.groupby("subject_id").size().min(), "to", meta.groupby("subject_id").size().max())
subj = meta.groupby("subject_id").agg(species=("species_name", "first"), sex=("sex", "first"),
                                      nspecies=("species_name", "nunique"), nsex=("sex", "nunique"))
print("subjects whose files disagree on species or sex:", int(((subj.nspecies > 1) | (subj.nsex > 1)).sum()))
print("subjects by species:", subj["species"].value_counts().to_dict())
print("subjects by sex:", subj["sex"].value_counts().to_dict())
print("sampling rates (Hz) of the sweeps used, files per set:",
      meta["rates"].value_counts().to_dict())
print("sweeps per file: median", meta["n_sweeps"].median(), "range", meta["n_sweeps"].min(),
      "to", meta["n_sweeps"].max())
print()
print("cell_id unique:", meta["cell_id"].is_unique)
print("files in the label table:", cells["Cortical_area"].notna().sum(), "of", len(cells),
      "| label table rows in the dataset:", tab["cell_name"].isin(meta["cell_id"]).sum(), "of", len(tab),
      "| lockdown table rows in the dataset:", lock["cell_name"].isin(meta["cell_id"]).sum(), "of", len(lock))
print("Cortical_area agrees with the code's region rule:",
      (cells["Cortical_area"] == cells["area_from_region"]).mean())
print("subject_id equals the table's Donor:", (cells["subject_id"] == cells["Donor"]).sum(), "of", len(cells))
print(cells.loc[cells["subject_id"] != cells["Donor"], ["cell_name", "subject_id", "Donor"]].to_string())
print("species (file) equals Species (table):", (cells["species_name"] == cells["Species"]).sum(), "of", len(cells))
print(cells.loc[cells["species_name"] != cells["Species"], ["cell_name", "subject_id", "species_name", "Species"]].to_string())
print()
print(pd.crosstab([cells["Cortical_area"], cells["Prep"]], cells["predicted_subclass"]))
# recordings deposited twice: identical feature vectors, or near-identical cell names
key = feats[FL].round(6).fillna(-9999).astype(str).agg("|".join, axis=1)
print("pairs of cells with identical feature vectors:", int(key.duplicated(keep=False).sum() // 2))
norm = meta["cell_id"].str.replace(r"\.0+(\d)", r".\1", regex=True)
dup = meta[norm.duplicated(keep=False)]
print("cell ids equal after removing leading zeros:", dup["cell_id"].tolist())

In [ ]:
# units and ranges of every feature used (mV, ms, pA, MOhm, Hz, V/s, Hz/pA)
rng_tab = feats[FL].describe(percentiles=[0.5]).T[["count", "min", "50%", "max"]]
print(rng_tab.round(3).to_string())
# clock check: spikes of the largest 1 s step fall inside the command epoch
print()
print("largest 1 s step: spikes inside the command epoch", int(feats["clk_in"].sum()),
      "| outside it", int(feats["clk_out"].sum()),
      "| cells with any spike outside", int((feats["clk_out"] > 0).sum()), "of", int(feats["clk_in"].notna().sum()))
print("latency of the first spike after onset (ms): median", round(feats["clk_lat_ms"].median(), 1),
      "| 5th to 95th percentile", feats["clk_lat_ms"].quantile([0.05, 0.95]).round(1).tolist())

## link-cells: the dataset's cells are the paper's cells

Claim (added by the author, protocol step 1.4): each file matches the paper's per-cell table by
cell name, the matched records agree on donor and species, and the features computed here from
the recordings agree with the table's values. Reproduced when every file matches a row, donor
and species agree for at least 99% of cells, and the median over the 39 features of the
Spearman correlation between the two is at least 0.9; partially reproduced when the match holds
but the agreement is lower. The 4 nA cells of the code's MCx.pkl and TCx.pkl are matched the
same way, and their spike counts compared step by step.

In [ ]:
rows = []
m = feats.merge(tab, on="cell_name", suffixes=("", "_paper"))
for f in FL:
    a, b = m[f].astype(float), m[f + "_paper"].astype(float)
    k = a.notna() & b.notna()
    rows.append(dict(feature=f, n=int(k.sum()), rho=stats.spearmanr(a[k], b[k])[0],
                     r=stats.pearsonr(a[k], b[k])[0],
                     median_rel_diff=float(np.median(np.abs(a[k] - b[k]) / np.abs(b[k]).clip(1e-9)))))
agree = pd.DataFrame(rows).set_index("feature")
print(agree.round(3).to_string())
print("median Spearman over features:", round(agree["rho"].median(), 3),
      "| features with rho >= 0.9:", int((agree["rho"] >= 0.9).sum()), "of", len(agree))
print("SFA here follows the Methods (second ISI over last); the table holds its inverse, rho =",
      round(agree.loc["SFA", "rho"], 3))

In [ ]:
MCX4 = d.zenodo_pickle("MCx.pkl")
TCX4 = d.zenodo_pickle("TCx.pkl")
ext = {x["meta"]["cell_id"]: x["ext"] for x in ok if x["ext"] is not None}
cmp = []
for area, pk in (("MCx", MCX4), ("TCx", TCX4)):
    for c, v in pk.items():
        e = ext.get(c)
        if e is None:
            cmp.append(dict(cell=c, area=area, found=False)); continue
        mm = pd.DataFrame(dict(i_inject=v["i_inject"], n_paper=v["num_spikes"])).merge(e, on="i_inject")
        cmp.append(dict(cell=c, area=area, found=True, steps=len(mm),
                        same_steps=bool(len(mm) == len(v["i_inject"]) == len(e)),
                        max_n_here=e["num_spikes"].max(), max_n_paper=int(np.max(v["num_spikes"])),
                        mad=float(np.median(np.abs(mm["num_spikes"] - mm["n_paper"])))))
cmp = pd.DataFrame(cmp)
print("4 nA cells in the dataset with a 200 pA series found:", cmp.groupby("area")["found"].agg(["sum", "count"]).to_dict())
print("cells whose steps match the pickle's exactly:", int(cmp["same_steps"].sum()), "of", len(cmp))
print("maximum spike count, here against the pickle: Spearman",
      round(stats.spearmanr(cmp["max_n_here"], cmp["max_n_paper"])[0], 3),
      "| cells within 10%:", int((np.abs(cmp["max_n_here"] - cmp["max_n_paper"]) <= 0.1 * cmp["max_n_paper"]).sum()))
print("median over cells of the median absolute step-wise difference in spike count:", cmp["mad"].median())

**Result: reproduced.** All 294 files match a row of the paper's table by cell name; 47 of the table's 341 rows are not in the dataset. Donor agrees for 293 of 294 cells and species for 292 of 294 (99.3%). The median Spearman correlation of the 39 features with the table's values is 0.952, and 27 of 39 features have rho of at least 0.9; SFA correlates at -0.989 because the table holds the inverse of the Methods' ratio. All 68 cells of MCx.pkl and TCx.pkl are in the dataset with a 200 pA series, but only 7 have exactly the same steps as the pickle; the maximum spike counts of 67 of 68 are within 10% of the pickle's (Spearman 0.996).

## cohort-animals

> Tissue came from 58 macaques, 23 male and 35 female, 48 Macaca nemestrina and 10 Macaca
> mulatta, aged 2 to 19 years.

In [ ]:
print("subjects with at least one file:", len(subj), "(expected 58, tolerance 5.8)")
print("male:", int((subj["sex"] == "M").sum()), "(23) | female:", int((subj["sex"] == "F").sum()),
      "(35) | unknown:", int((subj["sex"] == "U").sum()))
print("M. nemestrina:", int((subj["species"] == "M.nemestrina").sum()), "(48) | M. mulatta:",
      int((subj["species"] == "M.mulatta").sum()), "(10)")
def years(a):
    import re
    y = re.search(r"(\d+)Y", str(a)); return int(y.group(1)) if y else np.nan
ages = meta.groupby("subject_id")["age"].first().map(years)
print("age in years, min to max:", ages.min(), "to", ages.max(), "| subjects without an age:", int(ages.isna().sum()))

**Result: reproduced.** The files come from 57 subjects (expected 58 within 5.8): 22 male, 34 female and 1 of unknown sex, 48 M. nemestrina and 9 M. mulatta, all within tolerance. The youngest subject is recorded as 1 year old, where the paper says 2 to 19 years.

## cohort-recordings

> The physiology analysis rests on 333 recordings, 300 of them new in this study, plus 20 cells
> from a previous M1 study and 13 from a Patch-seq methods study.

In [ ]:
n = len(meta)
print("cells with current clamp data:", n)
print("(a) all 333: within 299.7 to 366.3:", 299.7 <= n <= 366.3)
print("(b) the 300 new cells: within 270 to 330:", 270 <= n <= 330)
print("the code's L5 table has", len(tab), "cells; those not in the dataset by recording user:",
      tab.loc[~tab["cell_name"].isin(meta["cell_id"]), "User"].value_counts().to_dict())

**Result: reproduced** under reading (b). The dataset holds 294 cells, within 270 to 330 of the 300 new cells and outside 299.7 to 366.3 for all 333. The 47 cells of the code's table that are missing were recorded mostly by one user (35 of 47).

## Physiology across subclasses (Figures 3 and 4)

The preprocessing of STAR Methods ("Physiology UMAP and classification"), on the 39 features
computed here (the code's 40 less zap_tau, which the paper does not define).

In [ ]:
pp, Z, n_drop = d.preprocess(cells[cells["Prep"].notna()], FL)
print("cells:", len(pp), "| excluded for more than 75% missing features:", n_drop)
print("missing values imputed:", int(cells.loc[pp.index, FL].isna().sum().sum()), "of", pp.shape[0] * len(FL))
print(pp.groupby(["Prep", "Cortical_area"]).size().to_dict())

### fig3a-cs-with-l5et

> In the UMAP of the physiology features, CS neurons lie only among the transcriptomic L5 ET
> neurons and overlap little with non-ET neurons, regardless of preparation, area and species.

In [ ]:
import umap
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
pca = PCA().fit(Z.values)
keep = pca.explained_variance_ratio_ >= 0.01
PCs = pca.transform(Z.values)[:, keep]
emb = umap.UMAP(n_neighbors=25, min_dist=0.1, metric="euclidean", random_state=20,
                n_components=2).fit_transform(PCs)
print("PCs kept:", int(keep.sum()), "explaining", round(pca.explained_variance_ratio_[keep].sum(), 3))
lab3 = np.asarray(pp["subclass_Corr_label"].fillna("none"), dtype=object)
cs = (pp["projection_type"] == "CS").values
ref = (lab3 != "none") & ~cs
print("CS cells:", int(cs.sum()), "| labelled non-CS cells:", int(ref.sum()),
      pd.Series(lab3[ref]).value_counts().to_dict())
def nn_rule(space, k):
    nnm = NearestNeighbors(n_neighbors=k).fit(space[ref])
    _, idx = nnm.kneighbors(space[cs])
    return (lab3[ref][idx] == "L5 ET")
ra = nn_rule(emb, 1)[:, 0]
rb = nn_rule(PCs, 10).sum(axis=1) >= 9
print("(a) 2-D: CS cells whose nearest labelled neighbour is L5 ET:", int(ra.sum()), "of", int(cs.sum()))
print("(b) PCs: CS cells with >= 9 of 10 L5 ET neighbours:", int(rb.sum()), "of", int(cs.sum()),
      "=", round(rb.mean(), 3), "(needs >= 0.9)")
sub = pp[cs].assign(a=ra, b=rb)
for g in ["Prep", "Cortical_area", "species_name"]:
    print(sub.groupby(g)[["a", "b"]].agg(["sum", "count"]).to_string())
fig, ax = plt.subplots(figsize=(5, 5))
for l in sorted(set(lab3)):
    k = (lab3 == l) & ~cs
    ax.scatter(emb[k, 0], emb[k, 1], s=8, label=l, alpha=0.7)
ax.scatter(emb[cs, 0], emb[cs, 1], marker="^", s=30, c="k", label="CS")
ax.legend(fontsize=7); ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
print("UMAP ranges:", emb.min(0).round(2), emb.max(0).round(2), "(all points shown)")

**Result: not reproduced.** In the two-dimensional embedding the nearest labelled non-CS neighbour of 46 of 52 CS neurons is L5 ET, not all of them (reading a), and in the space of the 20 retained PCs 18 of 52 CS neurons (0.346) have at least 9 of 10 L5 ET neighbours, against the 0.9 required (reading b). All 52 CS neurons are MCx and M. nemestrina, so the claim's independence of area and species cannot be tested; within preparations, reading (a) holds for 6 of 6 acute and 40 of 46 cultured CS neurons.

### fig3e-classifier-accuracy

> A logistic regression classifier trained on physiology predicts transcriptomic subclass (L5
> ET, L5 IT, L5/6 NP, L6 IT) with 92.8% accuracy weighted across subclasses on the whole dataset,
> 87.9% holding out 50% and 82.2% holding out 75%; for L5 ET it is 98.6% and 93.8% (75% held out).

In [ ]:
rng3e = np.random.default_rng(SEED + 1)
tr = pp[pp["subclass_Corr_label"].fillna("none") != "none"]
vc = tr["subclass_Corr_label"].value_counts()
dropped = vc[vc <= 5]
tr = tr[tr["subclass_Corr_label"].isin(vc[vc > 5].index)]
print("training cells:", len(tr), tr["subclass_Corr_label"].value_counts().to_dict(),
      "| classes dropped (<= 5 cells):", dropped.to_dict())
X3, y3 = Z.loc[tr.index].values, tr["subclass_Corr_label"].values
EXP3 = [92.8, 87.9, 82.2, 98.6, 93.8]
out3 = {}
for strat in (False, True):
    sc = {fr: d.classifier_scores(X3, y3, fr, 100, rng3e, stratify=strat) for fr in (1.0, 0.5, 0.25)}
    et = sc[1.0]["labels"].index("L5 ET")
    for rd, key, keyw in (("a f1", "f1", "f1_w"), ("b recall", "recall", "recall_w")):
        v = [100 * sc[1.0][keyw], 100 * sc[0.5][keyw], 100 * sc[0.25][keyw],
             100 * sc[1.0][key][et], 100 * sc[0.25][key][et]]
        within = [abs(a - b) <= 0.1 * b for a, b in zip(v, EXP3)]
        out3[(rd, strat)] = v
        print(f"reading ({rd}), stratified={strat}:", np.round(v, 1), "| within tolerance:", within)
    print("  per-class f1 at 100%, 50%, 25%:", {fr: dict(zip(sc[fr]["labels"], np.round(sc[fr]["f1"], 3))) for fr in sc})
print("chance (most frequent class share):", round(vc[vc > 5].max() / vc[vc > 5].sum(), 3))

**Result: reproduced, resting on an assumption.** With stratified random splits and the f1 reading (a), the five scores are 95.2, 79.2, 75.7, 99.5 and 91.7%, all within tolerance of 92.8, 87.9, 82.2, 98.6 and 93.8%. The 50% held-out score is 0.1 point above its lower bound (79.11). Without stratification it is 78.3%, outside the tolerance, under both score readings, and under the recall reading (b) the stratified 50% score is 79.1%, also outside. The classifier is trained on 167 cells (95 L5 ET, 39 L5 IT, 27 L6 IT, 6 L5/6 NP); the most frequent class is 56.9% of them.

### fig3-spc-classifier

> Classifiers built on sparse principal components of the physiology features perform similarly
> to those built on the features.

In [ ]:
from sklearn.decomposition import SparsePCA
rngsp = np.random.default_rng(SEED + 2)
spca = SparsePCA(n_components=8, random_state=0, alpha=7).fit(Z.values)
S = spca.transform(Z.values)
Xs = S[[Z.index.get_loc(i) for i in tr.index]]
for strat in (False, True):
    sc = {fr: d.classifier_scores(Xs, y3, fr, 100, rngsp, stratify=strat) for fr in (1.0, 0.5, 0.25)}
    et = sc[1.0]["labels"].index("L5 ET")
    for rd, key, keyw in (("a f1", "f1", "f1_w"), ("b recall", "recall", "recall_w")):
        v = [100 * sc[1.0][keyw], 100 * sc[0.5][keyw], 100 * sc[0.25][keyw],
             100 * sc[1.0][key][et], 100 * sc[0.25][key][et]]
        diff = np.array(v) - np.array(out3[(rd, strat)])
        print(f"sPC reading ({rd}), stratified={strat}:", np.round(v, 1), "| minus features:",
              np.round(diff, 1), "| within 10 points:", int((np.abs(diff) <= 10).sum()), "of 5")

**Result: reproduced.** On the 8 sparse PCs the five scores are within 10 percentage points of the feature-based scores under every reading and both split rules (largest difference 9.3 points, the whole-dataset weighted score).

### fig3h-projection-largest-effect

> In three-way ANOVAs of each physiological feature on projection type, preparation and cortical
> area, effect sizes are on average largest for projection type.

In [ ]:
c3 = cells[cells["predicted_subclass"].notna() & cells["Prep"].notna()].copy()
print("cells:", len(c3), c3.groupby(["projection_class", "Cortical_area", "Prep"]).size().to_dict())
A3 = d.anova_table(c3, FL, ["projection_class", "Prep", "Cortical_area"])
med = A3.groupby("term")[["eta_p", "eta_code"]].median().sort_values("eta_p", ascending=False)
print(med.round(4).to_string())
P_, PR_, AR_ = "C(projection_class)", "C(Prep)", "C(Cortical_area)"
print("median partial eta squared, projection > preparation and > area:",
      med.loc[P_, "eta_p"] > med.loc[PR_, "eta_p"] and med.loc[P_, "eta_p"] > med.loc[AR_, "eta_p"])
print("same with the code's formula:",
      med.loc[P_, "eta_code"] > med.loc[PR_, "eta_code"] and med.loc[P_, "eta_code"] > med.loc[AR_, "eta_code"])
print("mean partial eta squared:", A3.groupby("term")["eta_p"].mean().round(4).to_dict())

**Result: reproduced.** Over the 39 features the median partial eta squared is 0.067 for projection class, 0.021 for preparation and 0.008 for area; the code's formula gives the same order (0.054, 0.016, 0.006). The means give the same order (0.106, 0.038, 0.024).

### fig4a-top-subthreshold

> Among features differing between L5 ET and non-ET neurons, the largest effect sizes are for
> subthreshold, HCN-related properties, resonant frequency, input resistance and the 3 dB cutoff.

In [ ]:
pr = A3[(A3["term"] == P_) & (A3["p_bh"] <= 0.05)].sort_values("eta_p", ascending=False)
print("features with BH-corrected projection p <= 0.05:", len(pr), "of", len(FL))
print(pr[["feature", "eta_p", "eta_code", "p_bh"]].head(10).round(4).to_string(index=False))
top3, top5 = pr["feature"].head(3).tolist(), pr["feature"].head(5).tolist()
want = [{"Fr"}, {"MaxRn", "SSRn"}, {"cutoff"}]
print("top three:", top3, "| each of the three in the top three:", [bool(w & set(top3)) for w in want],
      "| in the top five:", [bool(w & set(top5)) for w in want])
prc = A3[(A3["term"] == P_) & (A3["p_bh"] <= 0.05)].sort_values("eta_code", ascending=False)
print("code's formula, top five:", prc["feature"].head(5).tolist())

**Result: not reproduced.** 26 of 39 features have a BH-corrected projection effect. The three largest are min dV/dt (0.381), min dV/dt of the 3 ms step spike (0.341) and maximum impedance (0.316); the 3 dB cutoff is eighth (0.261) and resonant frequency ninth (0.190), so none of the three named features is in the top three or the top five. The code's formula puts MaxZ first.

In [ ]:
def by_group(df, f, cls="projection_class", a="L5 ET", b="non L5 ET"):
    g = df.groupby(["Cortical_area", cls])[f].agg(["mean", "count"]).unstack(cls)
    g2 = df.groupby(["Prep", cls])[f].agg(["mean", "count"]).unstack(cls)
    return pd.concat([g, g2])
def proj_line(f):
    r = A3[(A3["term"] == P_) & (A3["feature"] == f)].iloc[0]
    return f"{f}: projection F = {r.F:.1f}, p = {r.p:.2g}, BH p = {r.p_bh:.2g}, partial eta^2 = {r.eta_p:.3f}"

### fig4c-et-higher-resonance

> L5 ET neurons have a higher resonant frequency than non-ET neurons in both the MCx and the TCx,
> and in both acute and cultured slices.

In [ ]:
print(proj_line("Fr"))
g = by_group(c3, "Fr"); print(g.round(2).to_string())
hi = (g[("mean", "L5 ET")] > g[("mean", "non L5 ET")])
print("L5 ET higher in each area and preparation:", hi.to_dict())
print("cells with the feature, L5 ET and non-ET:", c3.groupby("projection_class")["Fr"].count().to_dict(), "(paper 147 and 177)")

**Result: reproduced.** The projection effect on resonant frequency has BH-corrected p = 4e-14 (partial eta squared 0.190), and L5 ET neurons have the higher mean in MCx (5.37 against 2.31 Hz), TCx (3.33 against 1.79), acute (3.68 against 1.46) and cultured slices (4.85 against 2.34). The test has 132 L5 ET and 162 non-ET cells, against 147 and 177 in the paper.

### fig4e-et-lower-rin

> Maximum input resistance is lower in L5 ET than in non-ET neurons in both the MCx and the TCx,
> and in both preparations.

In [ ]:
print(proj_line("MaxRn"))
g = by_group(c3, "MaxRn"); print(g.round(1).to_string())
print("L5 ET lower in each area and preparation:", (g[("mean", "L5 ET")] < g[("mean", "non L5 ET")]).to_dict())
print("cells:", c3.groupby("projection_class")["MaxRn"].count().to_dict(), "(paper 145 and 173)")

**Result: reproduced.** The projection effect on maximum input resistance has BH-corrected p = 2.3e-7 (partial eta squared 0.097), and L5 ET neurons are lower in MCx (43.1 against 148.3 MOhm), TCx (59.7 against 163.2), acute (59.3 against 209.4) and culture (45.5 against 115.5), on 129 L5 ET and 159 non-ET cells (paper 145 and 173).

### fig4g-et-faster-spikes

> L5 ET neurons have faster action potential kinetics than non-ET neurons, with larger maximum and
> minimum dV/dt and a shorter half-width, and faster repolarization in both areas and preparations.

In [ ]:
for f, sign in (("max_dvdt", 1), ("min_dvdt", -1), ("AP_HW", -1)):
    print(proj_line(f))
    g = by_group(c3, f)
    faster = sign * (g[("mean", "L5 ET")] - g[("mean", "non L5 ET")]) > 0
    print(g.round(3).to_string()); print("  L5 ET faster:", faster.to_dict())
print("cells:", c3.groupby("projection_class")["AP_HW"].count().to_dict(), "(paper 148 and 176)")

**Result: reproduced.** The projection effects on max dV/dt (BH p = 4.9e-20), min dV/dt (5.9e-30) and half-width (9.4e-12) all have L5 ET faster, and min dV/dt is more negative in L5 ET within each area and preparation (MCx -137 against -105 V/s, TCx -119 against -76, acute -117 against -77, culture -135 against -90).

### By animal, L5 ET against non-ET

Within each donor with at least two cells of each class, the sign of the L5 ET minus non-ET
difference of the means.

In [ ]:
for f, sign in (("Fr", 1), ("MaxRn", -1), ("min_dvdt", -1), ("AP_HW", -1), ("max_dvdt", 1)):
    g = c3.groupby(["Donor", "projection_class"])[f].agg(["mean", "count"]).unstack()
    g = g[(g["count"] >= 2).all(axis=1)]
    dd = sign * (g[("mean", "L5 ET")] - g[("mean", "non L5 ET")])
    print(f"{f}: donors with both classes {len(g)}, L5 ET in the stated direction in {int((dd > 0).sum())}")

**Result: reproduced.** The projection effect on resonant frequency has BH-corrected p = 4e-14 (partial eta squared 0.190), and L5 ET neurons have the higher mean in MCx (5.37 against 2.31 Hz), TCx (3.33 against 1.79), acute (3.68 against 1.46) and cultured slices (4.85 against 2.34). The test has 132 L5 ET and 162 non-ET cells, against 147 and 177 in the paper.

### fig4-interaction-small

> Projection type by area interactions are small; no feature differs strongly between L5 ET and
> non-ET neurons in one area and not the other.

In [ ]:
IA = "C(projection_class):C(Cortical_area)"
ia = A3[A3["term"] == IA].set_index("feature")
medP = A3[A3["term"] == P_]["eta_p"].median()
print("median projection partial eta^2:", round(medP, 4), "| largest interaction:",
      ia["eta_p"].idxmax(), round(ia["eta_p"].max(), 4))
print("(a) every interaction below the median projection effect:", bool((ia["eta_p"] < medP).all()),
      "| features at or above it:", ia.index[ia["eta_p"] >= medP].tolist())
sig = ia[ia["p"] < 0.05].index.tolist()
print("interactions with uncorrected p < 0.05:", sig)
same = {}
for f in sig:
    g = c3.groupby(["Cortical_area", "projection_class"])[f].mean().unstack()
    dd = g["L5 ET"] - g["non L5 ET"]
    same[f] = bool(np.sign(dd["MCx"]) == np.sign(dd["TCx"]))
    print(f"  {f}: L5 ET minus non-ET, MCx {dd['MCx']:.3g}, TCx {dd['TCx']:.3g}, eta^2 {ia.loc[f, 'eta_p']:.3f}")
print("(b) same direction in both areas:", same, "| all:", all(same.values()))

**Result: not reproduced.** (a) One interaction, max_ifreq_gain (partial eta squared 0.080), is above the median projection effect (0.067). (b) Of the 15 features with an interaction at uncorrected p < 0.05, 4 (initial_ifreq_gain, max_ifreq_gain, SFA, f_AHP) differ between L5 ET and non-ET in opposite directions in the two areas. The other 38 interactions are below that median.

### species-no-effect

> A four-way ANOVA on species, preparation, area and subclass finds no significant effect of
> species on any of the 40 physiological features.

In [ ]:
c4 = c3[c3["species_name"].notna()].copy()
print("cells by species (from the files):", c4["species_name"].value_counts().to_dict())
for rd, cls in (("a", "projection_class"), ("b", "predicted_subclass")):
    try:
        A4 = d.anova_table(c4, FL, ["species_name", "Prep", "Cortical_area", cls])
        sp = A4[A4["term"] == "C(species_name)"]
        i = sp["p"].idxmin()
        print(f"({rd}) {cls}: features with BH species p < 0.05: {int((sp['p_bh'] < 0.05).sum())}"
              f" | uncorrected p < 0.05: {int((sp['p'] < 0.05).sum())} | smallest p {sp.loc[i, 'p']:.3g}"
              f" ({sp.loc[i, 'feature']}), BH {sp['p_bh'].min():.3g} | largest eta^2 {sp['eta_p'].max():.3f}")
    except Exception as e:
        print(f"({rd}) {cls}: the model could not be fitted:", repr(e)[:200])

**Result: not reproduced.** The four-way ANOVA finds a species effect under both readings of subclass: with projection class, 11 features at BH-corrected p < 0.05 (smallest p 5.9e-15, resting potential); with the predicted subclass, 1 feature (AP amplitude, BH p = 0.00047). The M. mulatta cells are 35 of 294 and come from 9 subjects, and the model with all interactions is rank deficient (some species, preparation, area and subclass cells are empty), so these p values rest on a partly unidentified design.

### tcx-et-types-no-difference

> Within TCx, physiological features do not differ significantly between L5 ET transcriptomic
> types (FDR-corrected p > 0.066), though some effect sizes are large.

In [ ]:
from statsmodels.stats.multitest import fdrcorrection
et = cells[(cells["Cortical_area"] == "TCx") & (cells["predicted_subclass"] == "L5 ET")
           & cells["cluster_Corr_label_MTG"].isin(["L5 ET_1", "L5 ET_2"])]
print("TCx L5 ET cells by type:", et["cluster_Corr_label_MTG"].value_counts().to_dict(), "(paper ET_1 25, ET_2 19)")
rows = []
for f in FL:
    a = et.loc[et["cluster_Corr_label_MTG"] == "L5 ET_1", f].dropna()
    b = et.loc[et["cluster_Corr_label_MTG"] == "L5 ET_2", f].dropna()
    rows.append(dict(feature=f, p=stats.ttest_ind(a, b, equal_var=False).pvalue, d=d.cohen_d(a, b)))
tt = pd.DataFrame(rows); tt["p_bh"] = fdrcorrection(tt["p"])[1]
i = tt["d"].abs().idxmax()
print("(a) Welch: features with BH p < 0.05:", int((tt["p_bh"] < 0.05).sum()), "| smallest BH p",
      round(tt["p_bh"].min(), 4), "| largest |d|", round(abs(tt.loc[i, "d"]), 2), tt.loc[i, "feature"])
A2t = d.anova_table(et.assign(ttype=et["cluster_Corr_label_MTG"]), FL, ["ttype", "Prep"])
ty = A2t[A2t["term"] == "C(ttype)"]
print("(b) two-way ANOVA type x preparation: features with BH p < 0.05:", int((ty["p_bh"] < 0.05).sum()),
      "| smallest BH p", round(ty["p_bh"].min(), 4), ty.loc[ty["p_bh"].idxmin(), "feature"])

**Result: reproduced.** On 21 L5 ET_1 and 17 L5 ET_2 TCx neurons (paper 25 and 19), no feature differs at BH-corrected p < 0.05 by Welch t-tests (smallest corrected p 0.060) or by the type effect of a two-way ANOVA with preparation (smallest 0.133). The largest effect size is d = 1.06 (train_current).

## L5 ET physiology between areas (Figure 6, S6)

Two-way ANOVAs, area by preparation, on the cells the code calls L5 ET (`predicted_subclass`).

In [ ]:
et6 = cells[(cells["predicted_subclass"] == "L5 ET") & cells["Prep"].notna()].copy()
print("L5 ET cells:", et6.groupby(["Cortical_area", "Prep"]).size().to_dict(),
      "| by area:", et6["Cortical_area"].value_counts().to_dict(), "(paper 77 or 78 MCx, 70 TCx)")
A6 = d.anova_table(et6, FL, ["Cortical_area", "Prep"])
AR = "C(Cortical_area)"
a6 = A6[A6["term"] == AR].set_index("feature")
means6 = et6.groupby("Cortical_area")[FL].mean().T
a6 = a6.join(means6)
def area_line(f):
    r = a6.loc[f]
    return (f"{f}: MCx {r.MCx:.3g}, TCx {r.TCx:.3g} (n = {int(r.n)}), area F = {r.F:.1f}, "
            f"p = {r.p:.2g}, BH p = {r.p_bh:.2g}, partial eta^2 = {r.eta_p:.3f} (code formula {r.eta_code:.3f})")

### fig6a-area-variation

> Two-way ANOVAs (preparation and area) show substantial cross-areal variation in subthreshold
> and suprathreshold properties of L5 ET neurons.

In [ ]:
SUB = ["Fr", "RQ", "cutoff", "Iphase", "phase0", "MaxZ", "MaxRn", "SSRn", "sag_ratio", "rebound", "rmp"]
sigA = a6[a6["p_bh"] < 0.05].sort_values("p_bh")
print("features with BH area p < 0.05:", len(sigA), "of", len(a6), "(the code's table: 19 of 40)")
print(sigA[["MCx", "TCx", "p_bh", "eta_p"]].round(4).to_string())
print("subthreshold among them:", [f for f in sigA.index if f in SUB])
print("suprathreshold among them:", [f for f in sigA.index if f not in SUB])
CODE19 = ["Fr", "cutoff", "zap_tau", "initial_ifreq_gain", "max_ifreq_gain", "up_down_ratio",
          "f_AHP", "MaxRn", "SSRn", "MaxZ", "AP_HW", "min_dvdt", "threshold_current"]
print("of the 13 the claims file names from the code's table, significant here:",
      [f for f in CODE19 if f in sigA.index], "| not:", [f for f in CODE19 if f not in sigA.index])

**Result: reproduced.** On 71 MCx and 61 TCx L5 ET neurons (paper 77 or 78 and 70), 16 of 39 features have a BH-corrected area effect (the code's table has 19 of 40), 5 subthreshold (cutoff, MaxRn, Fr, phase0, SSRn) and 11 suprathreshold (among them f_AHP, up_down_ratio and max_ifreq_gain). Of the 12 named features of the code's table computed here, 10 are significant; MaxZ and threshold_current are not.

### fig6b-mcx-higher-resonance

> L5 ET neurons in the MCx have a higher resonant frequency and a higher 3 dB cutoff than those
> in the TCx.

In [ ]:
for f in ("Fr", "cutoff"):
    print(area_line(f))
print("the code's means: fR 5.16 and 3.30 Hz, cutoff 10.99 and 8.10 Hz (MCx, TCx)")

**Result: reproduced.** Resonant frequency is 5.37 Hz in MCx and 3.33 Hz in TCx (BH p = 0.013) and the 3 dB cutoff 10.6 and 7.58 Hz (BH p = 0.0026), close to the code's means (5.16 and 3.30; 10.99 and 8.10).

### fig6d-mcx-deeper-fahp-symmetric

> L5 ET neurons in the MCx have a more pronounced fast AHP and faster, more symmetrical action
> potentials (lower ratio of max to min dV/dt) than those in the TCx.

In [ ]:
for f in ("f_AHP", "up_down_ratio", "AP_HW", "min_dvdt", "max_dvdt"):
    print(area_line(f))

**Result: reproduced.** In MCx the fAHP is deeper (-15.9 against -11.4 mV, BH p = 6.9e-5), the up/down ratio lower (2.69 against 3.31, BH p = 0.0013), the half-width shorter (0.629 against 0.741 ms, BH p = 0.035) and min dV/dt more negative (-137 against -119 V/s, BH p = 0.048). Max dV/dt is lower in MCx (356 against 376 V/s, BH p = 0.048), where the code found no difference.

### figs6b-mcx-lower-rin

> The input resistance of L5 ET neurons is lower in MCx than in TCx, in both acute and cultured
> slices.

In [ ]:
for f in ("SSRn", "MaxRn"):
    print(area_line(f))
    print(et6.groupby(["Prep", "Cortical_area"])[f].agg(["mean", "count"]).unstack().round(1).to_string())

**Result: reproduced.** Steady-state input resistance is lower in MCx (27.3 against 43.0 MOhm, BH p = 0.037) and lower in MCx within acute (30.8 against 51.4) and cultured slices (26.6 against 28.9, a small difference). Maximum input resistance, the variant, gives BH p = 0.0075, lower in MCx in both preparations.

### figs6b-mcx-higher-rheobase

> MCx L5 ET neurons need more current to start spiking than TCx L5 ET neurons.

In [ ]:
print(area_line("threshold_current"))

**Result: not reproduced.** Rheobase is higher in MCx (268 against 193 pA), but the area effect has p = 0.08 uncorrected and 0.16 after BH correction (partial eta squared 0.024), where the paper reports corrected p = 0.0496. On donor means (by-animal cell above) the difference has Welch p = 0.018, a variant that does not decide the outcome.

### figs6b-avg-gain-no-difference

> The slope of the average-rate FI curve (maximum gain) does not differ between MCx and TCx L5 ET
> neurons.

In [ ]:
print(area_line("max_gain"), "(Hz/pA)")

**Result: reproduced.** Maximum gain is 0.093 Hz/pA in MCx and 0.121 in TCx; the area effect has BH-corrected p = 0.59 (partial eta squared 0.003).

### fig6h-tcx-steeper-instantaneous-fi

> TCx L5 ET neurons fire onset doublets or bursts, so both the initial and the maximum slope of
> the maximum instantaneous FI curve are higher in TCx than in MCx.

In [ ]:
for f in ("initial_ifreq_gain", "max_ifreq_gain"):
    print(area_line(f), "(Hz/pA)")

**Result: reproduced.** Both slopes of the instantaneous FI curve are higher in TCx: initial 0.575 against 0.061 Hz/pA (BH p = 0.036) and maximum 1.05 against 0.33 Hz/pA (BH p = 0.0019).

### By animal, MCx against TCx L5 ET neurons

Donors give cells of one area, so the area comparison is repeated on donor means (Welch t-test).

In [ ]:
print("donors per area:", et6.groupby("Cortical_area")["Donor"].nunique().to_dict(),
      "| donors with cells in both areas:", int((et6.groupby("Donor")["Cortical_area"].nunique() > 1).sum()))
for f in ("Fr", "cutoff", "f_AHP", "up_down_ratio", "AP_HW", "min_dvdt", "SSRn", "threshold_current",
          "max_gain", "initial_ifreq_gain", "max_ifreq_gain"):
    dm = et6.groupby(["Cortical_area", "Donor"])[f].mean()
    t = stats.ttest_ind(dm["MCx"].dropna(), dm["TCx"].dropna(), equal_var=False)
    print(f"{f}: donor means MCx {dm['MCx'].mean():.3g} (n = {dm['MCx'].notna().sum()}), "
          f"TCx {dm['TCx'].mean():.3g} (n = {dm['TCx'].notna().sum()}), Welch p = {t.pvalue:.2g}")

**Result: reproduced.** On 71 MCx and 61 TCx L5 ET neurons (paper 77 or 78 and 70), 16 of 39 features have a BH-corrected area effect (the code's table has 19 of 40), 5 subthreshold (cutoff, MaxRn, Fr, phase0, SSRn) and 11 suprathreshold (among them f_AHP, up_down_ratio and max_ifreq_gain). Of the 12 named features of the code's table computed here, 10 are significant; MaxZ and threshold_current are not.

## Extended FI to 4 nA (Figure 6I to 6K, S6C, S6D)

The cells are the keys of the code's MCx.pkl and TCx.pkl; their series of 1 s steps by 200 pA is
read from the files.

In [ ]:
E = []
for area, pk in (("MCx", MCX4), ("TCx", TCX4)):
    for c in pk:
        if c in ext:
            E.append(ext[c].assign(cell=c, area=area))
over = {c: e.attrs.get("max_command_pA") for c, e in ext.items() if e.attrs.get("max_command_pA", 0) > 4000}
print("cells whose 200 pA series has commands above 4 nA (left out above 4 nA):", len(over),
      "| largest command, pA:", max(over.values()) if over else None)
E = pd.concat(E)
print("cells:", E.groupby("area")["cell"].nunique().to_dict(), "| steps per cell, range:",
      E.groupby("cell").size().min(), "to", E.groupby("cell").size().max(),
      "| largest step, pA, range:", E.groupby("cell")["i_inject"].max().min(), "to", E.groupby("cell")["i_inject"].max().max())
def welch(x, y):
    t = stats.ttest_ind(x, y, equal_var=False)
    return f"MCx {np.mean(x):.3g} (n = {len(x)}), TCx {np.mean(y):.3g} (n = {len(y)}), Welch p = {t.pvalue:.2g}, d (MCx - TCx) = {d.cohen_d(x, y):.2f}"
per = E.groupby(["area", "cell"])

In [ ]:
donor_of = dict(zip(cells["cell_name"], cells["Donor"]))
E["Donor"] = E["cell"].map(donor_of)
print("donors of the 4 nA cells:", E.groupby("area")["Donor"].nunique().to_dict())

### figs6c-mcx-higher-max-rate

> At maximal current injections MCx L5 ET neurons sustain higher average firing rates than TCx
> L5 ET neurons.

In [ ]:
mx = per["num_spikes"].max()
print("maximum spike count in a 1 s step:", welch(mx["MCx"], mx["TCx"]))
dmx = mx.rename(lambda c: donor_of.get(c, c), level=1).groupby(level=[0, 1]).mean()
print("on donor means:", welch(dmx["MCx"], dmx["TCx"]))
print("the code printed 101.6 MCx and 76.0 TCx, p 0.00024, d 1.00")

**Result: reproduced.** The maximum spike count in a 1 s step is 101 in MCx (42 cells) and 74.9 in TCx (26 cells), Welch p = 0.00015, d = 1.04, within tolerance of 1.0.

### figs6d-equal-max-instantaneous

> TCx L5 ET neurons reach higher instantaneous rates below 1 nA, but at up to 4 nA MCx L5 ET
> neurons fire at equivalent maximal instantaneous rates.

In [ ]:
mf = per["max_freq"].max()
print("maximum instantaneous rate over all steps (Hz):", welch(mf["MCx"], mf["TCx"]))
print("mean maximum instantaneous rate per step below 1 nA (Hz):")
print(E[E["i_inject"] < 1000].groupby(["i_inject", "area"])["max_freq"].mean().unstack().round(1).to_string())

**Result: reproduced.** The maximal instantaneous rate up to 4 nA does not differ (MCx 224 Hz, TCx 217 Hz, Welch p = 0.62, d = 0.11). Below 1 nA TCx neurons reach higher instantaneous rates at every step (for example 103.8 against 50.1 Hz at 800 pA), as the first clause says.

### fig6-depolarization-block

> At high current amplitudes, TCx L5 ET neurons undergo depolarization block far more often than
> MCx L5 ET neurons (14 of 26 against 1 of 42).

In [ ]:
def block_a(g):
    return bool((np.diff(g.sort_values("i_inject")["num_spikes"].values) < 0).any())
def block_b(g):
    n = g.sort_values("i_inject")["num_spikes"].values
    return bool(n[-1] < n.max())
ba, bb = per.apply(block_a), per.apply(block_b)
for nm, b in (("(a) any decrease", ba), ("(b) last step below the maximum", bb)):
    k = b.groupby("area").agg(["sum", "count"])
    print(nm, k.to_dict("index"), "| Fisher p =", round(stats.fisher_exact(
        [[k.loc["TCx", "sum"], k.loc["TCx", "count"] - k.loc["TCx", "sum"]],
         [k.loc["MCx", "sum"], k.loc["MCx", "count"] - k.loc["MCx", "sum"]]])[1], 6))
drop = per["num_spikes"].apply(lambda s: int(s.max() - s.values[-1]))
print("drop from the maximum to the last step, spikes; MCx:", drop["MCx"].describe()[["mean", "max"]].round(1).to_dict(),
      "TCx:", drop["TCx"].describe()[["mean", "max"]].round(1).to_dict())
for thr in (2, 5, 10):
    print(f"  (b) with a drop of more than {thr} spikes:", (drop > thr).groupby("area").sum().to_dict())

**Result: partially reproduced.** Block is more frequent in TCx, 17 of 26 neurons against 5 of 42 in MCx (Fisher p = 8e-6), and both readings give the same counts, but neither count is within tolerance of 14 of 26 and 1 of 42. Requiring a drop of more than 10 spikes from the curve's maximum gives 13 of 26 and 4 of 42.

In [ ]:
def at_max(g, f, code=False):
    g = g.sort_values("i_inject").reset_index(drop=True)
    n = g["num_spikes"].values
    if code:
        nz = np.nonzero(n)[0]
        k = int(np.argmax(n[nz]))  # ExtendedFI.ipynb indexes the full array with this
    else:
        k = int(np.argmax(n))
    return g.loc[k, f]

### fig6j-spike-width-ratio

> At the maximal average firing rate, spike width more than doubles from first to last spike in
> TCx L5 ET neurons but increases only 1.5 times in MCx L5 ET neurons.

In [ ]:
for rd, code_ in (("(a) step with the most spikes", False), ("(b) the code's indexing", True)):
    w = per.apply(lambda g: at_max(g, "width_ratio", code_)).dropna()
    print(rd, "half-width ratio, last to first:", welch(w["MCx"], w["TCx"]))
print("the code printed 2.27 TCx, 1.53 MCx, p 1.7e-9, d 2.02")

**Result: not reproduced.** At the step with the most spikes the last-to-first half-width ratio is 1.55 in TCx and 1.24 in MCx (Welch p = 2.9e-5, d = 1.28); with the code's indexing 1.53 and 1.26 (d = 1.20). The direction holds, but the TCx ratio does not exceed 2, the MCx ratio is outside 1.35 to 1.65 and d is outside 1.82 to 2.22 under both readings.

### fig6k-upstroke-ratio

> At the maximal average firing rate, max dV/dt of the last spike falls to about 20% of the first
> in TCx L5 ET neurons but only to about 50% in MCx.

In [ ]:
for rd, code_ in (("(a) step with the most spikes", False), ("(b) the code's indexing", True)):
    u = per.apply(lambda g: at_max(g, "upstroke_ratio", code_)).dropna()
    print(rd, "max dV/dt ratio, last to first:", welch(u["MCx"], u["TCx"]))
    du = u.rename(lambda c: donor_of.get(c, c), level=1).groupby(level=[0, 1]).mean()
    print("   on donor means:", welch(du["MCx"], du["TCx"]))
print("the code printed 0.221 TCx and 0.520 MCx, p 3.2e-10, d 1.69")

**Result: reproduced.** At the step with the most spikes the last-to-first max dV/dt ratio is 0.198 in TCx and 0.498 in MCx (Welch p = 2.5e-11, |d| = 1.79), all within tolerance of 0.2, 0.5 and 1.69; the code's indexing gives 0.229, 0.531 and 1.71.

## Irregular firing and subthreshold oscillations (Figure 7, S7)

The 10 s steps of the L5 ET cells (`predicted_subclass`). Each complete sweep is an observation.

In [ ]:
etcells = set(et6["cell_name"])
area_of = dict(zip(cells["cell_name"], cells["Cortical_area"]))
drug_of = dict(zip(cells["cell_name"], cells["drug"]))
SW, SEG = [], []
for x in ok:
    c = x["meta"]["cell_id"]
    if c not in etcells:
        continue
    for s in x["s10"]:
        t = s["t"]
        isi = np.diff(t)
        t100 = t[t >= 0.1]
        isi100 = np.diff(t100)
        rec = dict(cell=c, area=area_of[c], sweep=s["sweep"], amp=s["amp"], nsp=len(t),
                   cv=isi100.std() / isi100.mean() if len(isi100) >= 2 else np.nan,
                   max_isi=isi.max() if len(isi) else np.nan,
                   blockers=bool(isinstance(drug_of[c], str) and "dnqx" in drug_of[c].lower()))
        for bw in (1, 2):
            if len(isi):
                r = 1 / isi
                h = np.floor(r / bw)
                vals, cnt = np.unique(h, return_counts=True)
                rec[f"mode{bw}"] = (vals[np.argmax(cnt)] + 0.5) * bw
            else:
                rec[f"mode{bw}"] = np.nan
        fs_ = [g["fft_f"] for g in s["segs"]]
        rec["osc_f"] = np.mean(fs_) if fs_ else np.nan
        rec["nseg"] = len(fs_)
        SW.append(rec)
        for g in s["segs"]:
            SEG.append(dict(cell=c, area=area_of[c], nsp=len(t), blockers=rec["blockers"], **g))
SW, SEG = pd.DataFrame(SW), pd.DataFrame(SEG)
print("complete 10 s sweeps:", SW.groupby("area").agg(sweeps=("cell", "size"), cells=("cell", "nunique")).to_dict("index"))
print("spikes per sweep, range:", SW["nsp"].min(), "to", SW["nsp"].max(),
      "| sweeps with 1 to 200 spikes:", int(SW["nsp"].between(1, 200).sum()))
print("interspike segments longer than 650 ms:", len(SEG), SEG.groupby("area")["cell"].nunique().to_dict(),
      "| with an autocorrelogram frequency:", int(SEG["ac_f"].notna().sum()))
print("segment Vm (mV), range:", round(SEG["vm"].min(), 1), "to", round(SEG["vm"].max(), 1),
      "| FFT peak (Hz), range:", SEG["fft_f"].min(), "to", SEG["fft_f"].max(),
      "| share of segments with the peak at 3 to 3.5 Hz, the band's lower edge:", round((SEG["fft_f"] < 3.5).mean(), 3))

### figs7-cv-isi-higher-mcx

> During 10 s near-rheobase steps, MCx L5 ET neurons fire more irregularly (higher CV of ISIs)
> than TCx L5 ET neurons across a wide range of firing rates.

In [ ]:
cvd = SW[SW["nsp"] >= 10].dropna(subset=["cv"]).copy()
print("sweeps with >= 10 spikes:", cvd.groupby("area").agg(obs=("cell", "size"), cells=("cell", "nunique")).to_dict("index"),
      "(legend: MCx 273 from 64 cells, TCx 131 from 65)")
print("mean CV:", cvd.groupby("area")["cv"].mean().round(3).to_dict())
def shared_bins(df, bw, grp):
    """Spike-count bins holding at least 2 sweeps of each group (a two-way ANOVA with all
    interactions cannot be fitted on empty cells); returns the data and how many were dropped."""
    df = df.assign(bin=(df["nsp"] // bw).astype(int))
    n = df.groupby(["bin", grp]).size().unstack(fill_value=0)
    keep = n.index[(n >= 2).all(axis=1)]
    return df[df["bin"].isin(keep)], int((~df["bin"].isin(keep)).sum())
for lim in (None, 200):
    d0 = cvd if lim is None else cvd[cvd["nsp"] <= lim]
    for bw in (10, 25):
        dd, nd = shared_bins(d0, bw, "area")
        print(f"  ({nd} sweeps in bins without both areas left out)")
        t = d.anova(dd, "cv", ["area", "bin"])
        print(f"spikes <= {lim or 'any'}, bins of {bw}: area F = {t.loc['C(area)', 'F']:.1f}, "
              f"p = {t.loc['C(area)', 'PR(>F)']:.2g}, partial eta^2 = {t.loc['C(area)', 'eta_p']:.3f}, n = {t.attrs['n']}")

**Result: partially reproduced.** The CV of ISIs is higher in MCx (0.652 against 0.469, 267 sweeps from 63 cells against 126 from 57; legend 273 from 64 and 131 from 65), but the area effect has p = 0.010 with bins of 10 spikes and 0.017 with bins of 25, not below 0.001.

In [ ]:
def bins5(seg, col):
    s = seg.dropna(subset=[col]).assign(vbin=lambda x: np.floor(x["vm"] / 5) * 5)
    return s.groupby(["area", "cell", "vbin"]).agg(y=(col, "mean"), vm=("vm", "mean")).reset_index()
def corr_line(b, area):
    g = b[b["area"] == area]
    if len(g) < 3:
        return f"{area}: too few observations ({len(g)})"
    r, p = stats.pearsonr(g["vm"], g["y"])
    return f"{area}: r = {r:.3f}, r^2 = {r*r:.3f}, p = {p:.2g}, {len(g)} observations from {g['cell'].nunique()} cells"

### fig7-oscillation-frequency-vm

> The peak frequency of subthreshold oscillations between spikes increases with membrane
> potential in MCx L5 ET neurons but not in TCx, by autocorrelation and by Fourier analysis.

In [ ]:
for col, nm in (("ac_f", "autocorrelation"), ("fft_f", "FFT")):
    b = bins5(SEG, col)
    print(nm, "|", corr_line(b, "MCx"), "|", corr_line(b, "TCx"))
print("legend: autocorrelation MCx r^2 0.31 (79 obs, 52 cells), TCx 0.002 (66, 49); FFT MCx 0.32 (91, 59), TCx < 0.001 p 0.146 (82, 53)")

**Result: partially reproduced, resting on assumptions.** In MCx the oscillation frequency rises with membrane potential by autocorrelation (r = 0.53, r squared 0.284, p = 1.2e-5) and by FFT (r = 0.52, r squared 0.270, p = 2.9e-6). In TCx the FFT frequency shows no relation (p = 0.24), but the autocorrelation frequency falls with membrane potential (r = -0.32, p = 0.028), so one of the four conditions fails. The autocorrelation r squared is within tolerance of 0.31; the FFT r squared (0.270) is outside 0.288 to 0.352.

### fig7k-oscillation-power-vm

> Oscillation strength increases modestly with membrane potential, significantly in MCx L5 ET
> neurons and not significantly in TCx.

In [ ]:
b = bins5(SEG, "fft_p")
print("FFT strength |", corr_line(b, "MCx"), "|", corr_line(b, "TCx"))

**Result: not reproduced.** Oscillation strength does not rise with membrane potential in MCx (r = 0.057, p = 0.63, 72 observations); in TCx r = 0.20, p = 0.14.

### figs7-oscillations-with-blockers

> Subthreshold oscillations persist with fast synaptic transmission blocked, at a slightly reduced
> frequency, so they are intrinsic to the neuron.

In [ ]:
ob = SW.dropna(subset=["osc_f"])
print("sweeps with an oscillation frequency, by blockers:",
      ob.groupby("blockers").agg(sweeps=("cell", "size"), cells=("cell", "nunique")).to_dict("index"),
      "(legend: 55 cells with, 64 without)")
print("mean frequency (Hz):", ob.groupby("blockers")["osc_f"].mean().round(2).to_dict())
for bw in (10, 25):
    dd, nd = shared_bins(ob, bw, "blockers")
    print(f"  ({nd} sweeps in bins without both groups left out)")
    t = d.anova(dd, "osc_f", ["blockers", "bin"])
    print(f"bins of {bw}: blocker F = {t.loc['C(blockers)', 'F']:.2f}, p = {t.loc['C(blockers)', 'PR(>F)']:.2g}, n = {t.attrs['n']}")

**Result: partially reproduced.** Oscillations are found in cells recorded with synaptic blockers (140 sweeps from 42 cells), at a slightly lower mean frequency (5.43 against 5.58 Hz), but the blocker effect is not significant (p = 0.72 with bins of 10 spikes, 0.82 with bins of 25).

### fig7l-firing-tracks-oscillation

> In MCx L5 ET neurons the modal instantaneous firing rate during spiking epochs is well
> correlated with the subthreshold oscillation frequency; in TCx it is not.

In [ ]:
for bw in (1, 2):
    for area in ("MCx", "TCx"):
        g = SW.dropna(subset=["osc_f", f"mode{bw}"])
        g = g[g["area"] == area]
        r, p = stats.pearsonr(g["osc_f"], g[f"mode{bw}"])
        print(f"{bw} Hz bins, {area}: r = {r:.3f}, r^2 = {r*r:.3f}, p = {p:.2g}, {len(g)} sweeps from {g['cell'].nunique()} cells")
print("legend: MCx r^2 0.63 (122 obs, 42 cells), TCx < 0.001, p 0.683 (42, 29)")

**Result: partially reproduced.** In MCx the modal firing rate correlates with the oscillation frequency (r = 0.667, r squared 0.444, p = 3e-28, 209 sweeps from 53 cells; 0.454 with 2 Hz bins), below the tolerance of 0.63; in TCx it does not (p = 0.22 and 0.13).

### figs7g-modal-rates-long-isis

> MCx L5 ET neurons reach modal firing rates near 30 Hz while still having ISIs longer than 650
> ms, whereas no TCx neuron with such ISIs has a modal rate above about 8 Hz.

In [ ]:
# Figure S7G belongs to the oscillation analysis, which STAR Methods restricts to sweeps with
# at least one spike and no more than 200; all sweeps are shown as a variant
for nm, li in (("1 to 200 spikes", SW[(SW["max_isi"] > 0.65) & SW["nsp"].between(1, 200)]),
               ("any spike count (variant)", SW[SW["max_isi"] > 0.65])):
    for bw in (1, 2):
        print(f"{nm}, {bw} Hz bins: largest modal rate in sweeps with an ISI > 650 ms:",
              li.groupby("area")[f"mode{bw}"].max().to_dict(), "| sweeps:", li.groupby("area").size().to_dict())

**Result: reproduced, resting on an assumption.** Over sweeps with 1 to 200 spikes and an ISI longer than 650 ms, the largest modal rate is 30.5 Hz in MCx and 8.5 Hz in TCx with 1 Hz bins (31.0 and 9.0 Hz with 2 Hz bins, where TCx is above 8.8). Without the 200-spike limit the largest modal rates are 147.5 and 99.5 Hz.

## Morphology (data elsewhere: the paper's dendritic features)

The reconstructions are in the Brain Image Library; the paper's code deposited their features in
RawFeatureWide.csv (Zenodo record 13749343), which is used here.

In [ ]:
mo = d.zenodo_csv("RawFeatureWide.csv")
print("reconstructions:", mo["Cortical_Area"].value_counts().to_dict(), "(paper 12 MCx and 11 TCx)")
EXCL = ["specimen_id", "cell_name", "Cortical_Area", "apical_dendrite_depth_pc_0",
        "apical_dendrite_depth_pc_1", "apical_dendrite_depth_pc_2", "axon_exit_distance", "axon_exit_theta"]
MF = [c for c in mo.columns if c not in EXCL]
print(len(MF), "features (morphology.ipynb excludes the depth PCs and axon features)")
print("reconstructions whose cell is in the dandiset:", int(mo["cell_name"].isin(meta["cell_id"]).sum()))

### fig5-tap-root-dendrites

> Several MCx L5 ET neurons, including CS neurons, have long "tap root" basal dendrites reaching
> about 1.4 mm from the soma; TCx L5 ET neurons have none but have more complex apical tufts.

In [ ]:
for rd, f in (("(a) Euclidean", "basal_dendrite_max_euclidean_distance"), ("(b) path", "basal_dendrite_max_path_distance")):
    g = mo.groupby("Cortical_Area")[f]
    mcx = mo.loc[mo["Cortical_Area"] == "MCx", f]
    tcx = mo.loc[mo["Cortical_Area"] == "TCx", f]
    print(rd, f"MCx max {mcx.max()/1000:.3f} mm, median {mcx.median()/1000:.3f}; TCx max {tcx.max()/1000:.3f} mm;",
          "every TCx value below the MCx median:", bool((tcx < mcx.median()).all()))

**Result: reproduced.** On the paper's features of 10 MCx and 9 TCx reconstructions (paper 12 and 11), the largest basal Euclidean distance from the soma in MCx is 1.390 mm (reading a), within tolerance of 1.4 mm, and every TCx value (largest 0.686 mm) is below the MCx median (0.772 mm). The path distance reading gives 1.583 mm, outside the tolerance.

### fig5c-basal-apical-features

> MCx L5 ET neurons have longer basal dendrites extending farther horizontally, and TCx L5 ET
> neurons have more apical branches and a higher maximum apical branch order.

In [ ]:
rows = []
for f in MF:
    a = mo.loc[mo["Cortical_Area"] == "MCx", f].dropna()
    b = mo.loc[mo["Cortical_Area"] == "TCx", f].dropna()
    t = stats.ttest_ind(a, b, equal_var=False)
    rows.append(dict(feature=f, p=t.pvalue, d=d.cohen_d(a, b), MCx=a.mean(), TCx=b.mean()))
mt = pd.DataFrame(rows).dropna(subset=["p"])
mt["p_bh"] = fdrcorrection(mt["p"])[1]
show = ["basal_dendrite_max_path_distance", "basal_dendrite_total_length", "basal_dendrite_extent_x",
        "apical_dendrite_num_branches", "apical_dendrite_max_branch_order"]
print(mt.set_index("feature").loc[show].round(4).to_string())
print("paper: d 2.22 (basal length), 2.65 (horizontal extent), 1.56 (apical branches), 1.55 (max branch order)")

**Result: partially reproduced.** All four features differ at BH-corrected p < 0.05 in the stated direction with "longer" read as the maximum basal path distance (corrected p 0.0095, d 1.94), horizontal extent (0.0015, d 2.79), apical branches (0.017, d 1.64) and maximum apical branch order (0.0054, d 2.11). Total basal length does not differ (0.48). Two of the four d values are within tolerance (extent and branches); the others are not (1.94 against 2.22, 2.11 against 1.55).

### fig5e-morph-classifier

> A classifier predicts the area of an L5 ET neuron from dendritic features with 100% accuracy on
> the whole dataset, 91.9% weighted f1 holding out 50% and 76.1% holding out 75%.

In [ ]:
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
rng5 = np.random.default_rng(SEED + 5)
Xm = StandardScaler().fit_transform(KNNImputer(n_neighbors=2).fit_transform(mo[MF].astype(float)))
ym = mo["Cortical_Area"].values
for strat in (False, True):
    sc = {fr: d.classifier_scores(Xm, ym, fr, 100, rng5, stratify=strat) for fr in (1.0, 0.5, 0.25)}
    print(f"stratified={strat}: weighted f1 at 100%, 50%, 25% training:",
          [round(100 * sc[fr]["f1_w"], 1) for fr in sc], "| one-class draws redrawn:",
          [sc[fr]["redraws"] for fr in sc])
print("paper: 100, 91.9, 76.1; the deposited notebook printed 0.900 and 0.741")

**Result: partially reproduced.** Trained and tested on all 19 cells the classifier scores 100%. Holding out 50% it scores 78.0% (unstratified) or 80.3% (stratified), below 82.7, the lower bound for 91.9%; holding out 75%, 62.5% or 71.5%, so only the stratified split is within tolerance of 76.1%.

## Summary

| Claim | Outcome |
|---|---|
| link-cells | reproduced |
| cohort-animals | reproduced |
| cohort-recordings | reproduced (reading b, the 300 new cells) |
| patchseq-qc-177, cs-map-to-l5et, fig4h-patchseq-genes-et, patchseq-genes-area | blocked (other: Patch-seq reads in SRA, not processed in the time limit) |
| fig1-snrna-genes-et, fig1-snrna-genes-area | blocked (other: snRNA-seq references in NeMO, not processed in the time limit) |
| fig3a-cs-with-l5et | not reproduced |
| fig3e-classifier-accuracy | reproduced (assumed: stratified splits) |
| fig3-spc-classifier | reproduced |
| fig3h-projection-largest-effect | reproduced |
| fig4a-top-subthreshold | not reproduced |
| fig4c-et-higher-resonance | reproduced |
| fig4e-et-lower-rin | reproduced |
| fig4g-et-faster-spikes | reproduced |
| fig4-interaction-small | not reproduced |
| species-no-effect | not reproduced |
| tcx-et-types-no-difference | reproduced |
| fig5-tap-root-dendrites | reproduced |
| fig5b-dendrite-depth | blocked (other: needs the layer-aligned reconstructions) |
| fig5c-basal-apical-features | partially reproduced |
| fig5e-morph-classifier | partially reproduced |
| fig6a-area-variation | reproduced |
| fig6b-mcx-higher-resonance | reproduced |
| fig6d-mcx-deeper-fahp-symmetric | reproduced |
| figs6b-mcx-lower-rin | reproduced |
| figs6b-mcx-higher-rheobase | not reproduced |
| figs6b-avg-gain-no-difference | reproduced |
| fig6h-tcx-steeper-instantaneous-fi | reproduced |
| figs6c-mcx-higher-max-rate | reproduced |
| figs6d-equal-max-instantaneous | reproduced |
| fig6-depolarization-block | partially reproduced |
| fig6j-spike-width-ratio | not reproduced |
| fig6k-upstroke-ratio | reproduced |
| figs7-cv-isi-higher-mcx | partially reproduced |
| fig7-oscillation-frequency-vm | partially reproduced (assumed) |
| fig7k-oscillation-power-vm | not reproduced (assumed) |
| figs7-oscillations-with-blockers | partially reproduced (assumed) |
| fig7l-firing-tracks-oscillation | partially reproduced (assumed) |
| figs7g-modal-rates-long-isis | reproduced (assumed) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's:

- The paper's feature extraction was not deposited. All 39 features are computed here from the
  traces by the Methods' definitions, with standard choices where the Methods leave a detail open
  (helper docstrings). zap_tau, one of the code's 40 features, is not defined in the paper and is
  left out. SFA follows the Methods (second over last ISI); the paper's table holds the inverse.
- Files hold several runs of each protocol; which sweeps the paper used is not stated. The first
  run is used, and for the 1 s steps the first series reaching 1 nA.
- 294 of the 341 cells of the code's table are in the dataset, so every test has fewer cells
  than the paper's.
- The oscillation analysis (Figure 7) was R code that was not deposited. It is rebuilt from the
  Methods, with the standard choices the claims file names; the spline over segments is replaced
  by the mean over a sweep's segments, and the observations are cell by 5 mV bins.
- The morphology claims use the paper's deposited features of 19 reconstructions, not the 23
  reconstructions themselves.

What was found about the dataset:

- The files carry no area, preparation, subclass, drug or retrograde label; these come only from
  the code's tables, matched by cell name, which every file has.
- 47 cells of the code's L5 table (35 of them recorded by one user) are not in the dataset.
- One file's subject (Q19.26.009) differs from the table's donor (Q19.26.012), and the two files
  of subject Q19.01.001 are M. mulatta in the files and M. nemestrina in the table.
- One subject has sex U and one subject's age is 1 year, below the 2 to 19 years the paper
  reports.
- Some files hold 1 s command steps above 4 nA (up to 16 nA), beyond the 4 nA the Methods give
  as the headstage limit; they are left out here.